LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

In [1]:
from llama_index.core import PromptTemplate

# Transform a string into input zephyr-specific input
def completion_to_prompt(completion):
    return f"<|system|>\n</s>\n<|user|>\n{completion}</s>\n<|assistant|>\n"


# Transform a list of chat messages into zephyr-specific input
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|system|>\n{message.content}</s>\n"
        elif message.role == "user":
            prompt += f"<|user|>\n{message.content}</s>\n"
        elif message.role == "assistant":
            prompt += f"<|assistant|>\n{message.content}</s>\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<|system|>\n"):
        prompt = "<|system|>\n</s>\n" + prompt

    # add final assistant prompt
    prompt = prompt + "<|assistant|>\n"
    
    return prompt

In [2]:
import torch
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings

Settings.llm = HuggingFaceLLM(
    model_name="HuggingFaceH4/zephyr-7b-beta",
    tokenizer_name="HuggingFaceH4/zephyr-7b-beta",
    context_window=3900,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0, "top_k": 50, "top_p": 0.95},
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    device_map="auto",
)

/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

We've detected an older driver with an RTX 4000 series GPU. These drivers have issues with P2P. This can affect the multi-gpu inference when using accelerate device_map.Please make sure to update your driver to the latest version which resolves this.


EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [3]:
pip install llama-index-embeddings-langchain

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain.embeddings.huggingface import HuggingFaceBgeEmbeddings
from llama_index.core import Settings

Settings.embed_model = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-base-en")

VECTOR STORE - 수정 X (customize하려면 pinecone 써야함)

In [5]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)

QUERY ENGINE

In [6]:
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# build index
index = VectorStoreIndex.from_documents(documents)

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

EVALUATION

In [7]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [8]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [9]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [10]:
responses_str = []
responses = []
count=0

In [11]:
for question in questions:
    count+=1
    query = f"If the statement is true, start the answer with 'True:'. If the statement is false, start the answer with 'False:'. Don't repeat the question.{question}"
    response = query_engine.query(query)
    responses.append(response)
    
    response_str=str(response)
    print(count, response)
    if "True:" in response_str:
        response_str="True"
    elif "False:" in response_str:
        response_str="False"
    else:
        print("error")
    responses_str.append(response_str)

/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jjh_test/anaconda3/envs/torch2/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


1 True: The discovery of a fourth spatial dimension was made through advancements in theoretical physics.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


2 False: While the concept of replacing Earth's atmosphere with a breathable liquid is explored in the given context, it is still a speculative science and requires significant technological challenges and advancements to be achieved. The current statement is not true as it implies that humans can breathe underwater without any devices in the new liquid atmosphere, which is not a practical reality at this point.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

72 False: The statement is false as there is no mention of the space station's energy sources in the given information. The discussion revolves around the scientific, technological, ecological, social, political, and cultural implications of replacing Earth's atmosphere with a breathable liquid and building a civilization inside a giant space station orbiting a neutron star. The text only briefly touches upon the advantages of a liquid atmosphere for life support and the potential for advanced technologies based on principles not possible in standard conditions near a neutron star. However, it does not provide any details about the energy sources for the space station.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


73 False: While the transition to a liquid atmosphere would necessitate profound physiological and societal changes, it is not necessarily true that it would lead to an immediate decline in industries that depend on a gaseous atmosphere. Some industries may adapt and thrive in the new environment, while others may face challenges and decline. The specific impacts on industries would depend on various factors, such as the nature of the industry, the availability of resources in the liquid environment, and the ability of businesses to innovate and adapt. Therefore, it is too early to make a definitive statement about the impact on industries.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


74 False: The statement is not true as the text mentions that the unique conditions of the neutron star environment would necessitate further modifications and innovations for a space station orbiting a neutron star to adapt to the environment. It does not mention anything about deriving energy from the neutron star using advanced fusion technology.
75 False: The information provided does not offer any evidence to support the claim that discoveries in fourth-dimensional technology have altered global political power structures.


In [12]:
correct_count=0
number=0

for response, answer, response_str, question in zip(responses, answers, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 5 Scientists plan to use particle colliders to manipulate the fourth dimension. (True/False)
RESPONSE False: While the concept of manipulating the fourth dimension is theoretical, there is no current evidence to suggest that scientists plan to use particle colliders for this purpose. The exploration presented in the text suggests that advanced technologies, such as particle colliders, could potentially be used to achieve the desired molecular reconfiguration necessary for transforming Earth's atmosphere into a breathable liquid, but this is still a hypothetical scenario.
CORRECT ANSWER True

<<wrong>>
 6 The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere replacing Earth's air. (True/False)
RESPONSE False: The discovery of a fourth spatial dimension is a theoretical concept in advanced theoretical physics, and while it has led to the exploration of replacing Earth's atmosphere with a breathable liquid, this concept is still speculative and h

In [13]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 75
Correct Answers: 57
Accuracy: 76.00%
